# manual-chain-forward-and-back — faded example 1: Manual Forward and Backward Through exp → negate

> Faded drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `manual-chain-forward-and-back`. Running the beacon reports progress on the `Backprop: manual chain forward-and-back` subtopic.

**Most of the solution is filled in — complete the one blanked step**, run the test, then fire the beacon. Less scaffolding than the worked example, more than the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """A minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries an optional `.recipe`
    populated by wrap_forward_fn. `requires_grad` is set by the wrapper.
    `.grad` accumulates the leaf gradient at the end of the reverse pass."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
        self.grad = None
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: manual chain forward-and-back` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`manual-chain-forward-and-back`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "manual-chain-forward-and-back"
DD_SUBTOPIC = "Backprop: manual chain forward-and-back"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

For a chain `b = exp(a)`, `c = -b`, the backward runs in reverse: negate_back first, then exp_back. The negation backward simply multiplies `grad_out` by `-1`; the exp backward multiplies by the cached output (since `d/dx exp(x) = exp(x)`). Writing each step separately builds the habit of identifying which cached value each backward function depends on.

## Faded exercise 1

Complete the function below. The forward pass and both backward functions are defined. You must fill in the single blanked line that computes `dL/da` using `exp_back`.

Recall: `exp_back(grad_out, out, x) = grad_out * out` (the derivative of exp(x) equals exp(x) = out).

**Fill in:** Apply exp_back to compute dL/da from dL/db, using the cached output b.

In [ ]:
import torch as t

def negate_back(grad_out, out, x):
    """d/dx (-x) = -1"""
    return grad_out * (-1.0)

def exp_back(grad_out, out, x):
    """d/dx exp(x) = exp(x) = out; uses cached output"""
    return grad_out * out

def manual_exp_negate_chain(a, dL_dc):
    # Forward: a -> b -> c
    b = t.exp(a)
    c = -b
    # Backward (reverse order)
    dL_db = negate_back(dL_dc, c, b)  # already filled
    dL_da = None  # TODO: Apply exp_back to compute dL/da from dL/db, using the cached output b.
    return b, c, dL_db, dL_da

# Exercise it
t.manual_seed(31)
a_val = t.tensor([0.5, -1.0, 1.5])
dL_dc_val = t.ones(3)
b, c, dL_db, dL_da = manual_exp_negate_chain(a_val, dL_dc_val)
print(f"dL/da (manual): {dL_da}")


import torch as t

def _test():
    t.manual_seed(31)
    a_val = t.tensor([0.5, -1.0, 1.5])
    dL_dc_val = t.ones(3)

    b, c, dL_db, dL_da = manual_exp_negate_chain(a_val, dL_dc_val)

    # Verify against autograd
    a_ag = a_val.clone().requires_grad_(True)
    c_ag = -t.exp(a_ag)
    loss = (c_ag * dL_dc_val).sum()
    loss.backward()

    assert t.allclose(dL_da, a_ag.grad, atol=1e-5), \
        f"dL/da mismatch: manual={dL_da}, autograd={a_ag.grad}"
    assert t.allclose(dL_db, -dL_dc_val, atol=1e-5), \
        "dL/db should be -1 * dL_dc (negate backward)"


try:
    _test()
    _dd_passed.add('faded1')
    print('[Delta Drills] faded1 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report completion

Run the cell below to report progress. The beacon fires only if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
import torch as t

def negate_back(grad_out, out, x):
    """d/dx (-x) = -1"""
    return grad_out * (-1.0)

def exp_back(grad_out, out, x):
    """d/dx exp(x) = exp(x) = out; uses cached output"""
    return grad_out * out

def manual_exp_negate_chain(a, dL_dc):
    # Forward: a -> b -> c
    b = t.exp(a)
    c = -b
    # Backward (reverse order)
    dL_db = negate_back(dL_dc, c, b)
    dL_da = exp_back(dL_db, b, a)
    return b, c, dL_db, dL_da

# Exercise it
t.manual_seed(31)
a_val = t.tensor([0.5, -1.0, 1.5])
dL_dc_val = t.ones(3)
b, c, dL_db, dL_da = manual_exp_negate_chain(a_val, dL_dc_val)
print(f"dL/da (manual): {dL_da}")
```
</details>